# Notebook para exploracion y preparacion para Transformer

Este notebook se divide en dos bloques principales:

1. Exploracion inicial del dataset.
2. Analisis y preparacion de datos para entrenar el modelo

## 1) Exploracion inicial del dataset

En estas celdas se revisa:
- Carga del archivo.
- Vista inicial de registros.
- Estadisticas descriptivas.
- Valores nulos.
- Dimension del dataset y posibles duplicados.

Con esto validamos la calidad basica de los datos antes de modelar.


In [1]:
import pandas as pd


In [4]:
df = pd.read_csv("../data/train.csv")
df.head(10)

,text,label
0,Why has my transfer not arrived yet?,transfer_not_received_by_recipient
1,"Hae there, I have a transaction that is in pro...",pending_cash_withdrawal
2,"I tried to make a withdrawal from the ATM, but...",declined_cash_withdrawal
3,It is common for an international transfer to ...,balance_not_updated_after_bank_transfer
4,I am getting continuous failure for all my tra...,declined_transfer
5,Why do I get declined when I try to withdraw c...,declined_cash_withdrawal
6,is there a limit for top up?,top_up_limits
7,You promised no fees but now I've got one. Wha...,card_payment_fee_charged
8,I'm locked out of my account as I can't rememb...,passcode_forgotten
9,Why do you keep declining my payment?I tried s...,declined_card_payment


In [5]:
df.describe()

,text,label
count,10320,10320
unique,10320,77
top,Why has my transfer not arrived yet?,balance_not_updated_after_bank_transfer
freq,1,193


In [8]:
df.isna().sum()
df.isnull().sum()

text     0
label    0
dtype: int64

In [10]:
df.shape

(10320, 2)

In [11]:
df.drop_duplicates()
df.shape

(10320, 2)

In [15]:
def resumen_labels(df, text_col='text', label_col='label'):

    labels_disponibles = sorted(df[label_col].dropna().unique().tolist())
    conteo_textos_por_label = (
        df.groupby(label_col, dropna=False)[text_col]
        .count()
        .reset_index(name='cantidad_textos')
        .sort_values('cantidad_textos', ascending=False)
        .reset_index(drop=True)

    )
    return labels_disponibles, conteo_textos_por_label

labels, conteo = resumen_labels(df)
display(conteo)


,label,cantidad_textos
0,balance_not_updated_after_bank_transfer,193
1,cash_withdrawal_charge,193
2,transfer_fee_charged,191
3,wrong_exchange_rate_for_cash_withdrawal,187
4,reverted_card_payment?,187
...,...,...
72,compromised_card,87
73,card_acceptance,79
74,virtual_card_not_working,73
75,contactless_not_working,70


## 2) Preparacion para entrenamiento con Transformer

Desde aqui empieza la parte de analisis y preparacion para modelado:
- `resumen_labels`: muestra que clases existen y cuantos textos hay por clase.
- `label encoding`: convierte cada clase a un id numerico (`label_id`).
- `train/test split`: separa datos para entrenar y evaluar, manteniendo proporcion de clases.
- `tokenization`: transforma texto a tokens que entiende DistilBERT.
- `class weights`: ajusta el entrenamiento cuando hay desbalance entre clases.

Este bloque deja listo el dataset para la etapa de entrenamiento del modelo.

In [18]:
class LabelEncoder:
    def fit_transform(self, values):
        codes, classes = pd.factorize(values)
        self.classes_ = pd.Index(classes)
        return codes

text_col = 'text'
label_col = 'label'

label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df[label_col])

label_mapping = pd.DataFrame({
    'label_original': label_encoder.classes_,
    'label_id': range(len(label_encoder.classes_))
})

print('Mapping de labels:')
display(label_mapping)
df[[text_col, label_col, 'label_id']].head()

Mapping de labels:


,label_original,label_id
0,transfer_not_received_by_recipient,0
1,pending_cash_withdrawal,1
2,declined_cash_withdrawal,2
3,balance_not_updated_after_bank_transfer,3
4,declined_transfer,4
...,...,...
72,age_limit,72
73,card_about_to_expire,73
74,activate_my_card,74
75,top_up_reverted,75


,text,label,label_id
0,Why has my transfer not arrived yet?,transfer_not_received_by_recipient,0
1,"Hae there, I have a transaction that is in pro...",pending_cash_withdrawal,1
2,"I tried to make a withdrawal from the ATM, but...",declined_cash_withdrawal,2
3,It is common for an international transfer to ...,balance_not_updated_after_bank_transfer,3
4,I am getting continuous failure for all my tra...,declined_transfer,4


In [20]:
import numpy as np

def train_test_split(X, y, test_size=0.2, random_state=42, stratify=None):
    X = pd.Series(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    rng = np.random.default_rng(random_state)

    if stratify is None:
        idx = np.arange(len(X))
        rng.shuffle(idx)
        n_test = int(round(len(X) * test_size))
        test_idx = idx[:n_test]
        train_idx = idx[n_test:]
    else:
        stratify = pd.Series(stratify).reset_index(drop=True)
        train_idx = []
        test_idx = []

        for _, group_idx in stratify.groupby(stratify).groups.items():
            group_idx = np.array(list(group_idx))
            rng.shuffle(group_idx)

            if len(group_idx) <= 1:
                n_test = 0
            else:
                n_test = int(round(len(group_idx) * test_size))
                n_test = min(max(n_test, 1), len(group_idx) - 1)

            test_idx.extend(group_idx[:n_test])
            train_idx.extend(group_idx[n_test:])

        train_idx = np.array(train_idx)
        test_idx = np.array(test_idx)
        rng.shuffle(train_idx)
        rng.shuffle(test_idx)

    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

X = df[text_col].astype(str)
y = df['label_id']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')

Train size: 8256
Test size: 2064


In [ ]:
import sys
import subprocess

try:
    from transformers import AutoTokenizer
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])
    from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

print('Tokenizacion lista.')
print(f'Train batches tokenizados: {len(train_encodings["input_ids"])}')
print(f'Test batches tokenizados: {len(test_encodings["input_ids"])}')

C:\Users\males\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\males\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\males\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingfa

Tokenizacion lista.Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 1.6 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/10.4 MB 1.6 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.4 MB 1.3 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/10.4 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/10.4 MB 1.3 MB/s 

ERROR: Could not install packages due to an OSError: [WinError 5] Acceso denegado: 'C:\\Users\\males\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\tokenizers\\tokenizers.pyd'
Check the permissions.



In [ ]:

import numpy as np

def compute_class_weights_balanced(y_values):
    y_arr = np.asarray(y_values)
    classes, counts = np.unique(y_arr, return_counts=True)
    n_samples = len(y_arr)
    n_classes = len(classes)

    weights = n_samples / (n_classes * counts)
    return classes, weights

classes, weights = compute_class_weights_balanced(y_train)

try:
    import torch
    class_weights = torch.tensor(weights, dtype=torch.float)
    weights_view = class_weights.tolist()
    print('Class weights en tensor (torch)')
except ModuleNotFoundError:
    class_weights = weights
    weights_view = weights.tolist()
    print('Torch no instalado; class_weights queda como numpy array')

print('Class weights por label_id:')
for class_id, w in zip(classes, weights_view):
    print(f'label_id={class_id} -> weight={w:.4f}')

Class weights en tensor (torch)
Class weights por label_id:
label_id=0 -> weight=0.8861
label_id=1 -> weight=1.0021
label_id=2 -> weight=0.9324
label_id=3 -> weight=0.6962
label_id=4 -> weight=0.8578
label_id=5 -> weight=1.0410
label_id=6 -> weight=0.8248
label_id=7 -> weight=1.2918
label_id=8 -> weight=1.0310
label_id=9 -> weight=0.8789
label_id=10 -> weight=1.4688
label_id=11 -> weight=1.2764
label_id=12 -> weight=1.2468
label_id=13 -> weight=1.0310
label_id=14 -> weight=0.8717
label_id=15 -> weight=1.1054
label_id=16 -> weight=1.0115
label_id=17 -> weight=0.8443
label_id=18 -> weight=1.5317
label_id=19 -> weight=1.2184
label_id=20 -> weight=0.7344
label_id=21 -> weight=1.1169
label_id=22 -> weight=0.9405
label_id=23 -> weight=1.2468
label_id=24 -> weight=0.9747
label_id=25 -> weight=0.7826
label_id=26 -> weight=0.9405
label_id=27 -> weight=1.1406
label_id=28 -> weight=0.7148
label_id=29 -> weight=0.9405
label_id=30 -> weight=0.7148
label_id=31 -> weight=0.9660
label_id=32 -> weight=